# Reducción de Dimensionalidad sobre Matriz de Concurrencia
**Pipeline:** Corpus → Pre-procesamiento → Vocabulario → Concurrencia → PCA → t-SNE

## 1. Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import string # para código ASCII
import pandas as pd
import plotly.express as px

## 2. Tokenizador

In [2]:
# Tokenizador + limpiador del texto + eliminación de signos de puntuación

def tokenizador6(texto):
  texto_limpio = ""
  letras_permitidas = string.ascii_letters + "ñÑáéíóúÁÉÍÓÚ" + " "

  for caracter in texto:
    if caracter == "\n" or caracter == "\r" or caracter == "\t":
      texto_limpio += " "              # ← salto de línea → espacio
    elif caracter in letras_permitidas:
      texto_limpio += caracter 

  print(f"Texto después de limpiar (antes de stopwords): '{texto_limpio}'")
  # considerando mayúsculas y espacios
  token = ""
  tokens = []

  for i in range(len(texto_limpio)):
    caracter = texto_limpio[i]
    # Si encontramos un espacio separamos
    if caracter == ' ':
      if token != "":
        tokens = tokens + [token]
        token = ""
    else:
      token += caracter

  # Agregar el último token si existe
  if token != "":
    tokens = tokens + [token]

  return tokens


## 3. Corpus y pre-procesamiento

In [3]:
# conversión a minusculas 
def A_minusculas(texto):
    letras = ""

    for letra in texto:
        if letra == 'e':
            pass
        if ord(letra) >= 65 and ord(letra) <=90:
            letra = chr(ord(letra) +32)
        letras += letra
    return letras

# Removedor de stopwords general o removedor de stopwords de acuerdo al tema
def removedor_stop_words(tokenized_text):
    stop_words = ["de","la","que","el","en","y","a","los","del","se","las","por","un","para","con","una","su","al","lo"]
    new_text = []
    for item in tokenized_text:
        if item not in stop_words:
            new_text = new_text + [item]
    return new_text

def count_words(text):
    count = 0
    withes = string.whitespace + '.'
    a_word = False

    for char in text:
        if char not in withes :
            if not a_word:
                count += 1
                a_word = True
        else:
            a_word = False

    return count

In [4]:
lemmas_excepciones = {
    "fue":"ser",
    "fueron":"ser",
    "soy":"ser",
    "eres":"ser",
    "es":"ser",
    "estaba":"ser",
    "son":"ser",
    "malas":"malo",
    "malos":"malo",
    "buenas":"bueno",
    "peliculas":"pelicula",
    "actuaciones":"actuacion",
    "tramas":"trama",
    "mejores":"bueno",
    "era": "ser",
    "iba": "ir",
    "iban": "ir",
    "tuvo": "tener",
    "dijo" : "decir",
    "dijeron" : "decir",
    "dirán" : "decir",
    "hizo" : "hacer",
    "tuvo" : "tener",
    "árboles" : "árbol",
    "días" : "día"
}

diccionario_ar_gerundio = [
    "hablando", "cantando", "bailando", "saltando", "caminando","mirando", "escuchando", "pensando", "trabajando", "jugando",
    "estudiando", "viajando", "comprando", "pagando", "usando","tocando", "dibujando", "pintando", "imaginando", "recordando",
    "olvidando", "buscando", "llevando", "dejando", "guardando","explicando", "preguntando", "contestando", "cuidando", "ayudando",
    "intentando", "probando", "creando", "deseando", "esperando","logrando", "empezando", "terminando", "cambiando", "mejorando",
    "organizando", "planeando", "preparando", "presentando", "analizando","observando", "comparando", "señalando", "marcando", "considerando"
]

diccionario_ir_gerundio = [
    "viviendo","escribiendo","recibiendo","abriendo","permitiendo","admitiendo","asistiendo","dividiendo","decidiendo","repitiendo",
    "exigiendo","corrigiendo","dirigiendo","eligiendo","siguiendo","persiguiendo","consiguiendo","prohibiendo","imprimiendo","suprimiendo",
    "comprimiendo","expandiendo","confundiendo","difundiendo","fundiendo","hundiendo","interrumpiendo","cumpliendo","descubriendo","cubriendo",
    "inscribiendo","describiendo","suscribiendo","reescribiendo","proscribiendo","incluyendo","concluyendo","excluyendo","atribuyendo","distribuyendo",
    "retribuyendo","construyendo","destruyendo","instruyendo","sustituyendo","instituyendo","constituyendo","restituyendo","destituyendo","intuyendo","caminando"] #Verbos en infinitivo con termicación ir conjugados en gerundio

diccionario_er_gerundio = [
    "comiendo", "bebiendo", "leyendo", "corriendo", "temiendo","vendiendo", "aprendiendo", "entendiendo", "dependiendo", "sorbiendo",
    "mordiendo", "rompiendo", "respondiendo", "perdiendo", "volviendo","resolviendo", "envolviendo", "moviendo", "removiendo", "devolviendo",
    "cociendo", "torciendo", "retorciendo", "creciendo", "ofreciendo","mereciendo", "obedeciendo", "pareciendo", "estableciendo", "perteneciendo",
    "agradeciendo", "desapareciendo", "conociendo", "reconociendo", "traduciendo","produciendo", "reduciendo", "conduciendo", "introduciendo", "deduciendo",
    "seduciendo", "bendiciendo", "convenciendo", "venciendo", "defendiendo","encendiendo", "tendiendo", "extendiendo", "suspendiendo", "pretendiendo"
]

diccionario_ar_preterito = [
    "hablé", "hablaste", "habló", "hablamos", "hablaron",
    "canté", "cantaste", "cantó", "cantamos", "cantaron",
    "bailé", "bailaste", "bailó", "bailamos", "bailaron",
    "caminé", "caminaste", "caminó", "caminamos", "caminaron",
    "salté", "saltaste", "saltó", "saltamos", "saltaron",
    "miré", "miraste", "miró", "miramos", "miraron",
    "escuché", "escuchaste", "escuchó", "escuchamos", "escucharon",
    "trabajé", "trabajaste", "trabajó", "trabajamos", "trabajaron",
    "jugué", "jugaste", "jugó", "jugamos", "jugaron",
    "estudié", "estudiaste", "estudió", "estudiamos", "estudiaron",
    "viajé", "viajaste", "viajó", "viajamos", "viajaron",
    "compré", "compraste", "compró", "compramos", "compraron",
    "pagué", "pagaste", "pagó", "pagamos", "pagaron",
    "usé", "usaste", "usó", "usamos", "usaron",
    "toqué", "tocaste", "tocó", "tocamos", "tocaron",
    "dibujé", "dibujaste", "dibujó", "dibujamos", "dibujaron",
    "pinté", "pintaste", "pintó", "pintamos", "pintaron",
    "olvidé", "olvidaste", "olvidó", "olvidamos", "olvidaron",
    "busqué", "buscaste", "buscó", "buscamos", "buscaron",
    "ayudé", "ayudaste", "ayudó", "ayudamos", "ayudaron",
    "comencé", "comenzaste", "comenzó", "comenzamos", "comenzaron",
    "trabajé","trabajaste", "trabajó", "trabajamos", "trabajaron",
    "expliqué", "explicaste", "explicó", "explicamos", "explicaron",
    "cambié", "cambiaste", "cambió", "cambiamos", "cambiaron",
    "regresé", "regresaste", "regresó", "regresamos", "regresaron",
    "practiqué", "practicaste", "practicó", "practicamos", "practicaron",
    "preparé", "preparaste", "preparó", "preparamos", "prepararon",
    "organicé", "organizaste", "organizó", "organizamos", "organizaron",
    "terminé", "terminaste", "terminó", "terminamos", "terminaron",
    "mejoré", "mejoraste", "mejoró", "mejoramos", "mejoraron",
    "recordé", "recordaste", "recordó", "recordamos", "recordaron",
    "imaginé", "imaginaste", "imaginó", "imaginamos", "imaginaron",
    "pensé", "pensaste", "pensó", "pensamos", "pensaron"
]

diccionario_er_preterito = [
    "comí", "comiste", "comió", "comimos", "comieron",
    "bebí", "bebiste", "bebió", "bebimos", "bebieron",
    "corrí", "corriste", "corrió", "corrimos", "corrieron",
    "leí", "leíste", "leyó", "leímos", "leyeron",
    "temí", "temiste", "temió", "temimos", "temieron",
    "vendí", "vendiste", "vendió", "vendimos", "vendieron",
    "aprendí", "aprendiste", "aprendió", "aprendimos", "aprendieron",
    "entendí", "entendiste", "entendió", "entendimos", "entendieron",
    "dependí", "dependiste", "dependió", "dependimos", "dependieron",
    "sorbí", "sorbiste", "sorbió", "sorbimos", "sorbieron",
    "mordí", "mordiste", "mordió", "mordimos", "mordieron",
    "rompí", "rompiste", "rompió", "rompimos", "rompieron",
    "respondí", "respondiste", "respondió", "respondimos", "respondieron",
    "perdí", "perdiste", "perdió", "perdimos", "perdieron",
    "volví", "volviste", "volvió", "volvimos", "volvieron",
    "resolví", "resolviste", "resolvió", "resolvimos", "resolvieron",
    "moví", "moviste", "movió", "movimos", "movieron",
    "devolví", "devolviste", "devolvió", "devolvimos", "devolvieron",
    "torcí", "torciste", "torció", "torcimos", "torcieron",
    "ofrecí", "ofreciste", "ofreció", "ofrecimos", "ofrecieron",
    "respondí", "respondiste", "respondió", "respondimos", "respondieron",
    "nací", "naciste", "nació", "nacimos", "nacieron"
]

diccionario_ir_preterito = [
    "viví", "viviste", "vivió", "vivimos", "vivieron",
    "escribí", "escribiste", "escribió", "escribimos", "escribieron",
    "recibí", "recibiste", "recibió", "recibimos", "recibieron",
    "abrí", "abriste", "abrió", "abrimos", "abrieron",
    "permití", "permitiste", "permitió", "permitimos", "permitieron",
    "admití", "admitiste", "admitió", "admitimos", "admitieron",
    "asistí", "asististe", "asistió", "asistimos", "asistieron",
    "dividí", "dividiste", "dividió", "dividimos", "dividieron",
    "decidí", "decidiste", "decidió", "decidimos", "decidieron",
    "repetí", "repetiste", "repitió", "repetimos", "repitieron",
    "exigí", "exigiste", "exigió", "exigimos", "exigieron",
    "corrigí", "corregiste", "corrigió", "corregimos", "corrigieron",
    "dirigí", "dirigiste", "dirigió", "dirigimos", "dirigieron",
    "elegí", "elegiste", "eligió", "elegimos", "eligieron",
    "seguí", "seguiste", "siguió", "seguimos", "siguieron","seguía",
    "persiguí", "perseguiste", "persiguió", "perseguimos", "persiguieron",
    "conseguí", "conseguiste", "consiguió", "conseguimos", "consiguieron",
    "prohibí", "prohibiste", "prohibió", "prohibimos", "prohibieron",
    "imprimí", "imprimiste", "imprimió", "imprimimos", "imprimieron",
    "descubrí", "descubriste", "descubrió", "descubrimos", "descubrieron",
]

diccionario_ar_futuro = [
"hablaré","hablarás","hablará","hablaremos","hablarán",
"cantaré","cantarás","cantará","cantaremos","cantarán",
"bailaré","bailarás","bailará","bailaremos","bailarán",
"caminaré","caminarás","caminará","caminaremos","caminarán",
"saltaré","saltarás","saltará","saltaremos","saltarán",
"miraré","mirarás","mirará","miraremos","mirarán",
"escucharé","escucharás","escuchará","escucharemos","escucharán",
"trabajaré","trabajarás","trabajará","trabajaremos","trabajarán",
"jugaré","jugarás","jugará","jugaremos","jugarán",
"estudiaré","estudiarás","estudiará","estudiaremos","estudiarán",
"viajaré","viajarás","viajará","viajaremos","viajarán",
"compraré","comprarás","comprará","compraremos","comprarán",
"pagaré","pagarás","pagará","pagaremos","pagarán",
"usaré","usarás","usará","usaremos","usarán",
"tocaré","tocarás","tocará","tocaremos","tocarán",
"dibujaré","dibujarás","dibujará","dibujaremos","dibujarán",
"pintaré","pintarás","pintará","pintaremos","pintarán",
"buscaré","buscarás","buscará","buscaremos","buscarán",
"ayudaré","ayudarás","ayudará","ayudaremos","ayudarán",
"intentaré","intentarás","intentará","intentaremos","intentarán"
]

diccionario_er_futuro = [
"comeré","comerás","comerá","comeremos","comerán",
"beberé","beberás","beberá","beberemos","beberán",
"correré","correrás","correrá","correremos","correrán",
"leeré","leerás","leerá","leeremos","leerán",
"temeré","temerás","temerá","temeremos","temerán",
"venderé","venderás","venderá","venderemos","venderán",
"aprenderé","aprenderás","aprenderá","aprenderemos","aprenderán",
"entenderé","entenderás","entenderá","entenderemos","entenderán",
"dependeré","dependerás","dependerá","dependeremos","dependerán",
"morderé","morderás","morderá","morderemos","morderán",
"romperé","romperás","romperá","romperemos","romperán",
"responderé","responderás","responderá","responderemos","responderán",
"perderé","perderás","perderá","perderemos","perderán",
"volveré","volverás","volverá","volveremos","volverán",
"resolveré","resolverás","resolverá","resolveremos","resolverán",
"moveré","moverás","moverá","moveremos","moverán",
"devolveré","devolverás","devolverá","devolveremos","devolverán",
"torceré","torcerás","torcerá","torceremos","torcerán",
"ofreceré","ofrecerás","ofrecerá","ofreceremos","ofrecerán",
"temeré","temerás","temerá","temeremos","temerán"
]

diccionario_ir_futuro = [
"viviré","vivirás","vivirá","viviremos","vivirán",
"escribiré","escribirás","escribirá","escribiremos","escribirán",
"recibiré","recibirás","recibirá","recibiremos","recibirán",
"abriré","abrirás","abrirá","abriremos","abrirán",
"permitiré","permitirás","permitirá","permitiremos","permitirán",
"admitiré","admitirás","admitirá","admitiremos","admitirán",
"asistiré","asistirás","asistirá","asistiremos","asistirán",
"dividiré","dividirás","dividirá","dividiremos","dividirán",
"decidiré","decidirás","decidirá","decidiremos","decidirán",
"repetiré","repetirás","repetirá","repetiremos","repetirán",
"exigiré","exigirás","exigirá","exigiremos","exigirán",
"corregiré","corregirás","corregirá","corregiremos","corregirán",
"dirigiré","dirigirás","dirigirá","dirigiremos","dirigirán",
"elegiré","elegirás","elegirá","elegiremos","elegirán",
"seguiré","seguirás","seguirá","seguiremos","seguirán",
"perseguiré","perseguirás","perseguirá","perseguiremos","perseguirán",
"conseguiré","conseguirás","conseguirá","conseguiremos","conseguirán",
"prohibiré","prohibirás","prohibirá","prohibiremos","prohibirán",
"imprimiré","imprimirás","imprimirá","imprimiremos","imprimirán",
"descubriré","descubrirás","descubrirá","descubriremos","descubrirán"
]

def grammar_rules(word):
    n = len(word)
    
    if word[n-4:] == "ando" and word in diccionario_ar_gerundio:
        return word[:n-4] + "ar"
    if word[n-5:] == "iendo" and word in diccionario_ir_gerundio:
        return word[:n-5] + "ir"
    if word[n-5:] == "iendo" and word in diccionario_er_gerundio:
        return word[:n-5] + "er"

    # Pasado -ar
    if word[n-1:] == "é" and word in diccionario_ar_preterito:
        return word[:n-1] + "ar"
    if word[n-4:] == "aste" and word in diccionario_ar_preterito:
        return word[:n-4] + "ar"
    if word[n-1:] == "ó" and word in diccionario_ar_preterito:
        return word[:n-1] + "ar"
    if word[n-4:] == "amos" and word in diccionario_ar_preterito:
        return word[:n-4] + "ar"
    if word[n-4:] == "aron" and word in diccionario_ar_preterito:
        return word[:n-4] + "ar"

    # Pasado -er
    if word[n-1:] == "í" and word in diccionario_er_preterito:
        return word[:n-1] + "er"
    if word[n-2:] == "ía" and word in diccionario_er_preterito:
        return word[:n-2] + "er"
    if word[n-4:] == "iste" and word in diccionario_er_preterito:
        return word[:n-4] + "er"
    if word[n-2:] == "ió" and word in diccionario_er_preterito:
        return word[:n-2] + "er"
    if word[n-4:] == "imos" and word in diccionario_er_preterito:
        return word[:n-4] + "er"
    if word[n-5:] == "ieron" and word in diccionario_er_preterito:
        return word[:n-5] + "er"

    # Pasado -ir
    if word[n-1:] == "í" and word in diccionario_ir_preterito:
        return word[:n-1] + "ir"
    if word[n-2:] == "ía" and word in diccionario_ir_preterito:
        return word[:n-2] + "ir"
    if word[n-4:] == "iste" and word in diccionario_ir_preterito:
        return word[:n-4] + "ir"
    if word[n-2:] == "ió" and word in diccionario_ir_preterito:
        return word[:n-2] + "ir"
    if word[n-4:] == "imos" and word in diccionario_ir_preterito:
        return word[:n-4] + "ir"
    if word[n-5:] == "ieron" and word in diccionario_ir_preterito:
        return word[:n-5] + "ir"

    # Futuro -ar
    if word[-1:] == "é" and word in diccionario_ar_futuro:
        return word[:-1]
    if word[-2:] == "ás" and word in diccionario_ar_futuro:
        return word[:-2]
    if word[-1:] == "á" and word in diccionario_ar_futuro:
        return word[:-1]
    if word[-4:] == "emos" and word in diccionario_ar_futuro:
        return word[:-4]
    if word[-2:] == "án" and word in diccionario_ar_futuro:
        return word[:-2]

    # Futuro -er
    if word[-1:] == "é" and word in diccionario_er_futuro:
        return word[:-1]
    if word[-2:] == "ás" and word in diccionario_er_futuro:
        return word[:-2]
    if word[-1:] == "á" and word in diccionario_er_futuro:
        return word[:-1]
    if word[-4:] == "emos" and word in diccionario_er_futuro:
        return word[:-4]
    if word[-2:] == "án" and word in diccionario_er_futuro:
        return word[:-2]

    # Futuro -ir
    if word[-1:] == "é" and word in diccionario_ir_futuro:
        return word[:-1]
    if word[-2:] == "ás" and word in diccionario_ir_futuro:
        return word[:-2]
    if word[-1:] == "á" and word in diccionario_ir_futuro:
        return word[:-1]
    if word[-4:] == "emos" and word in diccionario_ir_futuro:
        return word[:-4]
    if word[-2:] == "án" and word in diccionario_ir_futuro:
        return word[:-2]
    

def lematizador1(corpus):
    lematized_words = []
    for word in corpus:
        lematized_word = grammar_rules(word)
        if lematized_word == None and word in lemmas_excepciones:
            lematized_word = lemmas_excepciones[word]
        elif lematized_word == None and word not in lemmas_excepciones:
            lematized_word = word
        lematized_words += [lematized_word]
    return lematized_words

In [5]:
# Función para leer el contenido de un documento
def leer_documento(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def preprocesar_texto(texto):
    # 1. minúsculas
    texto = A_minusculas(texto)
    
    # 2. tokenización + limpieza
    tokens = tokenizador6(texto)
    
    # 3. stopwords
    tokens = removedor_stop_words(tokens)
    
    # 4. lematización
    tokens = lematizador1(tokens)
    
    return tokens

In [6]:
# Corpus con los documentos y sus respectivas clases
corpus = [
    ("doc1.txt", leer_documento("doc1.txt"), "politica")
]

n_docs = len(corpus)
print(n_docs)

1


In [7]:
corpus_procesado = []

i = 0
while i < n_docs:
    path, texto, clase = corpus[i]
    
    tokens = preprocesar_texto(texto)
    
    corpus_procesado += [(tokens, clase)]
    i += 1

print(corpus_procesado[0])


Texto después de limpiar (antes de stopwords): ' discurso de vicente fox como candidato presidencial del pan hoy más allá de las cifras se expresa la voluntad política de la sociedad de nuestro país libremente sin acarreos y con una profunda voluntad de cambio en la elección democrática de acuerdo a estatutos del candidato del pan a la presidencia de la república un partido que nunca ha pretendido acaparar ni manipular como otros sino ser el más consciente y comprometido con el cambio que méxico requiere amigos y amigas vengo con una gran emoción a compartir con ustedes esta fiesta ciudadana celebrada en todos los rincones del país por mi partido acción nacional y en la que hemos participado miles y miles de mexicanos al elegirme como candidato de nuestro partido a la presidencia de la república me conmueve profundamente el honor que me han otorgado el más importante de mi vida y que me ofrece la oportunidad de servir a mi país con la pasión con la que hace doce años me comprometí al a

## 4. Vocabulario (palabras únicas)

In [8]:
vocabulario = []

i = 0
while i < len(corpus_procesado):
    tokens, _ = corpus_procesado[i]
    
    j = 0
    while j < len(tokens):
        palabra = tokens[j]
        
        # evitar duplicados
        existe = False
        k = 0
        while k < len(vocabulario):
            if vocabulario[k] == palabra:
                existe = True
                break
            k += 1
        
        if not existe:
            vocabulario += [palabra]
        
        j += 1
    
    i += 1

vocab_size = len(vocabulario)

print("Tamaño del vocabulario:", vocab_size)

Tamaño del vocabulario: 247


## 5. Matriz de Concurrencia

In [9]:
# Aplanar todos los tokens del corpus en una sola secuencia
doc_tokenizado = []
i = 0
while i < len(corpus_procesado):
    tokens, _ = corpus_procesado[i]
    j = 0
    while j < len(tokens):
        doc_tokenizado += [tokens[j]]
        j += 1
    i += 1

# Diccionario palabra → índice (necesario para indexar la matriz)
palabra_a_idx = {}
i = 0
while i < len(vocabulario):
    palabra_a_idx[vocabulario[i]] = i
    i += 1

def construir_concurrencia(vocabulario, doc_tokenizado, palabra_a_idx, ventana=2):
    n = len(vocabulario)
    matriz_conc = np.zeros((n, n))

    i = 0
    while i < len(doc_tokenizado):
        palabra_actual = doc_tokenizado[i]
        if palabra_actual not in palabra_a_idx:
            i += 1
            continue
        idx_actual = palabra_a_idx[palabra_actual]

        inicio = i - ventana
        if inicio < 0:
            inicio = 0
        fin = i + ventana + 1
        if fin > len(doc_tokenizado):
            fin = len(doc_tokenizado)

        j = inicio
        while j < fin:
            if j != i:
                vecino = doc_tokenizado[j]
                if vecino in palabra_a_idx:
                    idx_vecino = palabra_a_idx[vecino]
                    matriz_conc[idx_actual][idx_vecino] += 1
            j += 1

        i += 1

    return matriz_conc


matriz_conc = construir_concurrencia(vocabulario, doc_tokenizado, palabra_a_idx, ventana=2)

print("Forma de la matriz de concurrencia:", matriz_conc.shape)
print("(", vocab_size, "palabras ×", vocab_size, "palabras )")
print("\nEjemplo — co-ocurrencias de '", vocabulario[0], "':")
j = 0
while j < vocab_size:
    if matriz_conc[0][j] > 0:
        palabra_vecina = vocabulario[j]
        conteo = int(matriz_conc[0][j])
        print(palabra_vecina, ":", conteo)
    j += 1


Forma de la matriz de concurrencia: (247, 247)
( 247 palabras × 247 palabras )

Ejemplo — co-ocurrencias de ' discurso ':
vicente : 1
fox : 1


In [10]:
matriz_conc

array([[0., 1., 1., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.],
       [1., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 1., 1.],
       [0., 0., 0., ..., 1., 0., 1.],
       [0., 0., 0., ..., 1., 1., 0.]], shape=(247, 247))

## 6. PCA — Reducción de dimensionalidad

Cada palabra pasa de ser un vector de **V dimensiones** (su fila en la matriz de concurrencia)  
a un vector de **N componentes principales** que conserva la mayor varianza posible.

In [11]:
N_COMPONENTES_PCA = 60  # no puede superar vocab_size (actualmente 247)

pca = PCA(n_components=N_COMPONENTES_PCA)
embeddings_pca = pca.fit_transform(matriz_conc)

print("Forma ANTES de PCA :", matriz_conc.shape)
print("Forma DESPUÉS de PCA:", embeddings_pca.shape)
print("\nVarianza explicada por componente:")
i = 0
while i < N_COMPONENTES_PCA:
    print("  PC" + str(i+1) + ": " + str(round(pca.explained_variance_ratio_[i]*100, 2)) + "%")
    i += 1
print("  TOTAL acumulado:", round(pca.explained_variance_ratio_.sum()*100, 2), "%")


Forma ANTES de PCA : (247, 247)
Forma DESPUÉS de PCA: (247, 60)

Varianza explicada por componente:
  PC1: 6.94%
  PC2: 4.83%
  PC3: 3.88%
  PC4: 3.07%
  PC5: 2.94%
  PC6: 2.58%
  PC7: 2.42%
  PC8: 2.3%
  PC9: 2.26%
  PC10: 2.08%
  PC11: 1.91%
  PC12: 1.89%
  PC13: 1.74%
  PC14: 1.65%
  PC15: 1.52%
  PC16: 1.41%
  PC17: 1.38%
  PC18: 1.32%
  PC19: 1.3%
  PC20: 1.25%
  PC21: 1.23%
  PC22: 1.12%
  PC23: 1.12%
  PC24: 1.08%
  PC25: 1.02%
  PC26: 0.98%
  PC27: 0.97%
  PC28: 0.92%
  PC29: 0.89%
  PC30: 0.87%
  PC31: 0.85%
  PC32: 0.83%
  PC33: 0.8%
  PC34: 0.77%
  PC35: 0.76%
  PC36: 0.73%
  PC37: 0.72%
  PC38: 0.69%
  PC39: 0.68%
  PC40: 0.68%
  PC41: 0.64%
  PC42: 0.62%
  PC43: 0.61%
  PC44: 0.6%
  PC45: 0.59%
  PC46: 0.59%
  PC47: 0.57%
  PC48: 0.55%
  PC49: 0.54%
  PC50: 0.52%
  PC51: 0.52%
  PC52: 0.52%
  PC53: 0.51%
  PC54: 0.5%
  PC55: 0.49%
  PC56: 0.47%
  PC57: 0.46%
  PC58: 0.46%
  PC59: 0.45%
  PC60: 0.45%
  TOTAL acumulado: 77.03 %


In [12]:
df_var = pd.DataFrame({
    "Componente": list(range(1, N_COMPONENTES_PCA + 1)),
    "Varianza acumulada": np.cumsum(pca.explained_variance_ratio_)
})

fig = px.line(df_var, x="Componente", y="Varianza acumulada",
              markers=True, title="PCA — Varianza explicada acumulada")
fig.add_hline(y=0.80, line_dash="dash", line_color="red",
              annotation_text="80%", annotation_position="bottom right")
fig.show()

## 7. t-SNE — Visualización en 2D

t-SNE **no** es para entrenar: es exclusivamente para **visualizar**.  
Toma los embeddings de PCA (ya de menor dimensión) y los proyecta a 2D  
de forma que palabras similares queden cerca en el plano.

In [13]:
perplejidad = vocab_size - 1
if perplejidad > 30:
    perplejidad = 30

tsne = TSNE(n_components=2, perplexity=perplejidad, random_state=42, max_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings_pca)

print("Forma de los embeddings 2D (t-SNE):", embeddings_2d.shape)


Forma de los embeddings 2D (t-SNE): (247, 2)


In [14]:
df_tsne = pd.DataFrame({
    "x": embeddings_2d[:, 0],
    "y": embeddings_2d[:, 1],
    "palabra": vocabulario
})

fig = px.scatter(df_tsne, x="x", y="y", text="palabra",
                 title="t-SNE — embeddings PCA sobre matriz de concurrencia")
fig.update_traces(textposition="top center", textfont_size=9)
fig.show()

## 8. Resultado: diccionario palabra → embedding

In [15]:
embedding_por_palabra = {}
i = 0
while i < vocab_size:
    embedding_por_palabra[vocabulario[i]] = embeddings_pca[i]
    i += 1

palabra_ejemplo = vocabulario[0]
print("Embedding de '", palabra_ejemplo, "':")
print(embedding_por_palabra[palabra_ejemplo])
print("\nDimension del embedding:", embedding_por_palabra[palabra_ejemplo].shape)
print("\nTotal de palabras con embedding:", len(embedding_por_palabra))


Embedding de ' discurso ':
[-2.81828875e-01 -1.53588936e-01 -1.06320839e-01 -6.06870444e-03
 -3.81576994e-02 -1.65997339e-01 -2.91989202e-02 -3.23182437e-02
 -2.39999110e-02 -8.18081254e-02 -2.13563374e-01  2.10692522e-01
 -5.22385023e-02 -1.69307696e-01  4.27399107e-02  7.84812143e-02
 -1.15697224e-02 -3.74324503e-02 -2.30084865e-02  8.80844436e-02
  1.88533776e-01 -8.31912464e-02  1.16373455e-01  1.25335191e-01
  1.25122341e-01 -7.22397012e-02  1.71287907e-01  2.08163191e-03
 -3.10468910e-02  6.07302827e-02  1.26147583e-01  6.53628137e-02
 -1.74958058e-02 -1.01451487e-01  5.53337256e-02 -8.84077679e-02
 -1.23208786e-01  2.16289748e-04  6.36991311e-03  1.11598409e-01
 -9.23168765e-02  4.05963503e-04  4.28447619e-02 -1.69449175e-01
 -1.06077416e-01 -1.60023283e-02 -1.69970587e-01  4.02427367e-01
  2.25141744e-01 -5.25046706e-02  1.84907226e-01  2.31040251e-01
 -2.85941839e-02  3.52781628e-01  5.14669311e-02  3.61814388e-02
 -6.97488468e-05  4.03627492e-02  1.12063100e-02 -1.11020008e-0